In [0]:
import os
# Install the required packages from the specified requirements file
os.system("pip install -r https://raw.githubusercontent.com/George-Michael-Dagogo/World_news_tutorial/main/requirements.txt")
# Import necessary libraries
from newsapi.newsapi_client import NewsApiClient
import pandas as pd
from newspaper import Article, Config
from nltk.corpus import stopwords
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datetime import date, timedelta

In [0]:
def extract_transform_function():
    today = date.today()
    #get today's date
    yesterday = today - timedelta(days = 1)
    #get yesterday's date
    day_before_yesterday = today - timedelta(days=2)
    #get day before yesterday's date
    #initialize news api client with an api key
    newsapi = NewsApiClient(api_key='ff4373852c2343a98303951439854f8c')

    #get top headlines for the entertainment category in English,with a page size of 90
    top_headlines = newsapi.get_top_headlines(
        category = 'entertainment',
        language = 'en',
        page_size = 90,
        page = 1
    )
    #Extract articles from the API response
    articles = top_headlines.get('articles',[])

    #Create a DataFrame from the articles, selecting specific columns
    init_df = pd.DataFrame(articles,columns=['source','title','publishedAt','author','url'])

    # Extract the 'name' field from the 'source' dictionary in each row
    init_df['source'] = init_df['source'].apply(lambda x: x['name'] if pd.notna(x) and 'name' in x else None)

    # Convert 'publishedAt' to datetime format
    init_df['publishedAt'] = pd.to_datetime(init_df['publishedAt'])

    # Filter the DataFrame for articles published on the day before yesterday or yesterday
    filtered_df = init_df[(init_df['publishedAt'].dt.date == day_before_yesterday) | (init_df['publishedAt'].dt.date == yesterday)]
    # Rename the 'publishedAt' column to 'date_posted'
    filtered_df.rename(columns={'publishedAt': 'date_posted'}, inplace=True)
    # Make a copy of the filtered DataFrame
    df = filtered_df.copy()
    def full_content(url):
        # Set up the user agent for the browser configuration
        user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36'
        config = Config()
        config.browser_user_agent = user_agent
        page = Article(url, config = config)

        try:
            # Download and parse the article
            page.download()
            page.parse()
            return page.text
        except Exception as e:
            print(f"Error retrieving content from {url}: {e}")
            return 'couldnt retrieve'
        
    # Apply the full_content function to each URL in the DataFrame
    df['content'] = df['url'].apply(full_content)
    # Replace newlines in the 'content' column with spaces
    df['content'] = df['content'].str.replace('\n', ' ')
    # Filter out rows where the content could not be retrieved
    df = df[df['content'] != 'couldnt retrieve']
    # Download the NLTK stopwords dataset and other required datasets
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('wordnet')

    # Function to count words in a text excluding stopwords
    def count_words_without_stopwords(text):
        if isinstance(text, (str, bytes)):
            words = nltk.word_tokenize(str(text))
            stop_words = set(stopwords.words('english'))
            filtered_words = [word for word in words if word.lower() not in stop_words]
            return len(filtered_words)
        else:
            return 0

    # Apply the word count function to the 'content' column
    df['word_count'] = df['content'].apply(count_words_without_stopwords)
    # Download the VADER sentiment analysis lexicon
    nltk.download('vader_lexicon')

    # Initialize the SentimentIntensityAnalyzer
    sid = SentimentIntensityAnalyzer()

    # Function to get sentiment and compound score for a given text
    def get_sentiment(row):
        sentiment_scores = sid.polarity_scores(row)
        compound_score = sentiment_scores['compound']

        if compound_score >= 0.05:
            sentiment = 'Positive'
        elif compound_score <= -0.05:
            sentiment = 'Negative'
        else:
            sentiment = 'Neutral'

        return sentiment, compound_score

    # Apply the sentiment analysis function to the 'content' column
    df[['sentiment', 'compound_score']] = df['content'].astype(str).apply(lambda x: pd.Series(get_sentiment(x)))

    return df

# Call the extract_transform_function and store the result in a DataFrame
dataframe = extract_transform_function()

display(dataframe)


/home/spark-531b9b29-ae61-4996-938c-9f/.ipykernel/2084/command-4876407047160265-677956533:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.rename(columns={'publishedAt': 'date_posted'}, inplace=True)


Error retrieving content from https://www.usatoday.com/story/entertainment/tv/2026/05/10/roast-of-kevin-hart-netflix-tom-brady-kevin-harts-wife/90024796007/: Article `download()` failed with 403 Client Error: Unknown Error for url: https://www.usatoday.com/story/entertainment/tv/2026/05/10/roast-of-kevin-hart-netflix-tom-brady-kevin-harts-wife/90024796007/ on URL https://www.usatoday.com/story/entertainment/tv/2026/05/10/roast-of-kevin-hart-netflix-tom-brady-kevin-harts-wife/90024796007/
Error retrieving content from https://ew.com/the-boys-showrunner-reacts-to-show-predicting-future-trump-gold-statue-unveiled-11971184: Article `download()` failed with 402 Client Error: Payment Required for url: https://ew.com/the-boys-showrunner-reacts-to-show-predicting-future-trump-gold-statue-unveiled-11971184 on URL https://ew.com/the-boys-showrunner-reacts-to-show-predicting-future-trump-gold-statue-unveiled-11971184
Error retrieving content from https://abcnews.com/GMA/Family/sandra-bullock-shar

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/spark-531b9b29-ae61-4996-938c-9f/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/spark-531b9b29-ae61-4996-938c-9f/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/spark-531b9b29-ae61-4996-938c-9f/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/spark-531b9b29-ae61-4996-938c-9f/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


source title date_posted author url content word_count sentiment compound_score TMZ Pete Davidson Jokes That Kanye West is a 'Gay Nazi' During Kevin Hart Roast - TMZ 2026-05-11T09:15:41.000Z TMZ Staff https://www.tmz.com/2026/05/11/pete-davidson-kanye-west-kevin-hart/ If you thought the beef between Pete Davidson and Kanye West was over, then think again ... because Pete may have just poured gasoline on the fire. Here's the deal ... Pete came out on stage at the Kia Forum in L.A. Sunday night to roast Kevin Hart ... but he also cracked jokes about the late Charlie Kirk, Tony Hinchcliffe, Lizzo and, of course, Kanye. Waiting for your permission to load the Twitter Tweet. Aiming below the belt, Pete let it rip, "I was in a beef with Kanye, so I’ve taken shots from better gay Nazis.” Ouch! The comedian was clearly poking fun at Kanye for fully embracing Hitler and the Nazis over several years beginning in 2022. Recently, the rapper made apologies for his antisemitic remarks, even taking out a full-page ad in the Wall Street Journal to say he was sorry. As everyone knows, Kanye went after Pete publicly during Pete's short-lived relationship with Kanye's ex-wife, Kim Kardashian, also in 2022. Kanye mocked Pete online, giving him a derogatory nickname, "Skete," and posting music videos showing Kanye inflicting violence on Pete. As for Pete's gay crack, it's confusing ... after all, Kanye is married to the voluptuous Bianca Censori. 162 Positive 0.7537 Whereisthebuzz.com Sydney Sweeney Goes Full Godzilla in ‘Euphoria’ Season 3 and the Internet Is Not OK - Yahoo 2026-05-11T06:10:00.000Z Talia M. https://whereisthebuzz.com/sydney-sweeney-goes-full-godzilla-in-euphoria-season-3-and-the-internet-is-not-ok/ A scene from the third season of HBO Max’s series “Euphoria” became quite controversial online after a fantasy scene featuring Sydney Sweeney garnered considerable attention. In the episode, Sweeney portrays Cassie Howard, who has become famous for creating adult content with her partner, Maddy Perez. During the fantasy scene, Cassie grows in size and becomes as tall as characters from the movies Attack of the 50 Foot Woman and the Godzilla franchise; then she destroys parts of Los Angeles, all while helicopters surround her. The character Rue Bennett says, “She knew this was her destiny to triumph, to conquer, to win.” Social media users have reacted negatively to the scene, with many calling it a “humiliation ritual”. As one user put it, “How many more fetishes can they fit into this show? This has to be a record.” Also, the show’s plot is now being viewed as drifting far from its original intentions. “They don’t talk about Rue’s addiction anymore, isn’t that why the show is called ‘Euphoria’?” one Twitter user asked. The Euphoria creator, Sam Levinson, has received much criticism, with one Twitter user asking: “What the h— did Sam Levinson do to this series?” Another user commented saying: “Sam Levinson is making straight up f—— p–n and he needs to be stopped.” However, the controversy occurs amid a wider discussion of the treatment of actresses in the movie industry. In a recent interview with The Independent, Sydney Sweeney described some of the criticism of her nudity. “I’m very proud of my work in Euphoria,” Sweeney said. “I thought it was a great performance. But no one talks about it because I got naked.” In turn, Sam Levinson defended her, saying that her acting was brilliant, according to The Hollywood Reporter. Euphoria Season 3 is currently airing on HBO Max. 238 Positive 0.9431 Forbes ‘Project Hail Mary’ Arrives On Streaming This Week As Film Tops $655 Million At Box Office - Forbes 2026-05-11T06:00:20.000Z Tim Lammers https://www.forbes.com/sites/timlammers/2026/05/11/project-hail-mary-arrives-on-streaming-this-week-as-film-tops-655-million-at-box-office/ "Project Hail Mary" key art featuring Ryan Gosling. Amazon MGM Studios Ryan Gosling’s Project Hail Mary is new on digital streaming this week as the blockbuster space a

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS the_news;
CREATE TABLE IF NOT EXISTS the_news.news_table(
    source STRING,
    title STRING,
    date_posted DATE,
    author STRING,
    url STRING,
    content STRING,
    word_count INT,
    sentiment STRING,
    compound_score DOUBLE
)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, DoubleType

# Initialize Spark session
spark = SparkSession.builder.appName("CreateTableExample").getOrCreate()

# Define the schema explicitly if necessary
schema = StructType([
    StructField("source", StringType(), True),
    StructField("title", StringType(), True),
    StructField("date_posted", DateType(), True), 
    StructField("author", StringType(), True),
    StructField("url", StringType(), True),
    StructField("content", StringType(), True),
    StructField("word_count", IntegerType(), True),
    StructField("sentiment", StringType(), True),
    StructField("compound_score", DoubleType(), True)
])

spark_df = spark.createDataFrame(dataframe, schema=schema)
spark_df.write.mode('overwrite').saveAsTable('the_news.news_table')

In [0]:
%sql
select sentiment, count(*) from the_news.news_table group by sentiment

sentiment,count(*)
Positive,27
Negative,7
